day06

0. 복습
SVM
 : 종류 사이의 간격(마진)을 가장 넓게 벌리는 경계를 찾는 분류 모델
- 마진 : 경계선과 가장 가까운 데이터 사이의 간격
- 서포트 벡터 : 마진의 가장자리에 딱 닿은 데이터들

1) 커널(kernel)
 : 곡선으로 데이터를 나눠야 할때 사용하는 기술

2) C와 gamma
- c : 오류 허용 
- gamma : RBF 영향 범위 

1. 랜덤 포레스트(Random Forest)
 : 조금씩 다르게 학습한 결정나무 여러 그루의 예측을 모아
   다수결로 결정하는 모델
- 나무 한 그루는 실수할 수 있지만, 서로 다른 실수를 하는
  나무 여럿의 의견을 모으면 실수가 상쇄되어 더 정확하고
  안정적이다
- 이렇게 여러 모델을 모아 더 좋은 성능을 내는 방법을 통틀어
  앙상블(ensemble, 집단학습)이라 부른다.
- 랜덤 포레스트는 대표적인 앙상블 중 하나

1) 앙상블과 배깅
- 나무 한 그루의 예측은 불안정하다.
- 학습 데이터가 조금만 달라져도 가지가 완전히 다르게 갈라질 수 있다
- 그래서 랜덤 포레스트는 나무를 여러 그루 만들되,
  각 나무에게 조금씩 다른 데이터를 준다.
- 전체 데이터에서 무작위로 일부를 뽑아 나무 마다 다른 훈련 세트를
  만든다
- 이 방식을 배깅(bagging)이라 한다. 그리고 각 나무의 예측을 모아
  다수결로 최종 예측을 한다

2) "랜덤"의 의미
- 나무를 다르게 만드는 법 
- 랜덤 포레스트는 나무들을 서로 다르게 만들기 위해
  무작위를 두 군데에 넣는다
(1) 데이터(행) : 나무마다 훈련 데이터를 무작위로 다시 뽑음
                 (배깅)
		 => 나무마다 본 데이터가 다름
(2) 특성(열) : 각 질문에서 쓸 특성 후보를 일부만 무작위로
 	       골라 그 중 최적의 선택
	       => 나무마다 쓰는 질문이 다름
- 그냥 배깅만 하면 나무들이 강한 특성부터 질문하게 되어
  서로 비슷해진다
- 그래서 각 질문마다 특성 후보를 일부만 랜덤으로 보여주면,
  어떤 나무는 성별을, 어떤 나이를 먼저 물어보게 되어
  진짜 다양한 나무들이 만들어진다

3) 하이퍼파라미터
 : 랜덤 포레스트에서 자주 조절하는 값들
(1) n_estimators : 나무수
		나무를 몇 그루를 심을지.
		많을수록 안정되지만
		계산이 느려진다(보통 100 ~ 300)
(2) max_depth : 나무 최대 깊이
		각 나무를 얼마나 깊게 키울지
		랜덤 포레스트는 깊어도 과적합에 강한편
(3) max_features : 질문마다 볼 특성 수
		작을수록 나무가 더 다양해 진다
		(보통 기본값이면 충분)

4) 정리
- 랜덤포레스트 : 조금씩 다른 결정나무 여러 그루의 투표(다수결)
	대표적인 앙상블(집단학습) 모델
- 최대 장점 : 나무 하나보다 과적합에 강하고 안정적
	      특성 중요도도 고르게 나온다

## 랜덤 포레스트
> 타이타닉 승객의 생존 여부를 예측하는 랜덤 포레스트 모델

In [ ]:
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset('titanic')
titanic.head()

In [ ]:
df = titanic[['survived', 'pclass', 'sex', 'age', 'fare']]

# 전처리
# 1) 성별을 숫자로 변환
df['sex'] = df['sex'].map({"male" : 0, "female" : 1})

# 2) 나이의 결측치를 중앙값으로 대체
# print(df.isnull().sum())
df['age'] = df['age'].fillna(df['age'].median())

# 특성(독립변수)/ 타깃(종속변수) 분할
X = df[['pclass', 'sex', 'age', 'fare']]
y = df['survived']

df.head()

### 랜덤 포레스트 모델 생성, 학습, 예측, 평가

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier # 랜덤포레스트 모델
from sklearn.metrics import accuracy_score, classification_report

# 훈련/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 랜덤 포레스트 모델 생성(나무 100그루짜리 숲)
# n_estimators : 나무 수, random_state : 결과 고정용
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train) 
# 학습 : 100그루의 나무를 각각 다른 데이터로 학습

# 예측(테스트 데이터 사용)
pred = forest.predict(X_test) # 예측 : 100그루가 투표해 다수결로 결정

# 평가 
print(f"정확도 : {accuracy_score(y_test, pred) : .3f}")
print(classification_report(y_test, pred))
# 정확도는 0.793이 나온다
# 이 지표만 보면 결정 나무와 큰 차이가 없어 보인다

### 결정 나무 vs 랜덤 포레스트

In [ ]:
# 결정 나무는 깊이를 제한하지 않으면(끝까지 키우면) 과적합 문제가
# 발생할 수 있다
# 나무 하나와 숲을 똑같이 제한 없이 키워 비교
from sklearn.tree import DecisionTreeClassifier # 결정나무 모델

# 1) 결정나무 한 그루(깊이 제한 없음 -> 끝까지 자람)
one_tree = DecisionTreeClassifier(random_state=42)
one_tree.fit(X_train, y_train)

# 2) 랜덤 포레스트(나무 100그루, 깊이 제한 없음)
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

# 학습용 정확도와 평가용 정확도를 각각 계산(둘의 차이가 과적합 정도)
# 1) 나무 1그루 - 학습용 
tree_train = accuracy_score(y_train, one_tree.predict(X_train))
# 2) 나무 1그루 - 평가용
tree_test = accuracy_score(y_test, one_tree.predict(X_test))
# 3) 숲 - 학습용
forest_train = accuracy_score(y_train, forest.predict(X_train))
# 4) 숲 - 평가용
forest_test = accuracy_score(y_test, forest.predict(X_test))

# 비교
print(f"결정나무 1그루 | 학습용 {tree_train : .3f} / 평가용 {tree_test : .3f}")
print(f"랜덤포레스트 | 학습용 {forest_train : .3f} / 평가용 {forest_test : .3f}")

# 학습용 정확도는 둘다 0.978로 똑같다. 둘다 학습 데이터는 거의 외움
# 하지만 처음 보는 평가용 데이터에서 갈린다
# - 나무 한 그루 학습 데이터의 사소한 특성까지 외워새 데이터에 약하다
# - 반면 숲은 서로 다르게 과적합한 나무들이 투표하면서 각자의 실수가 상쇄되면서
#   새 데이터에도 비교적 안정적이다
# ** 랜덤 포레스트의 장점 - 깊이를 일일이 조절하지 않아도 과적합에 강하다

### 특성 중요도

In [ ]:
importance = pd.DataFrame({
    "특성" : ['객실등급', '성별', '나이', '요금'],
    "중요도" : forest.feature_importances_.round(3)
}).sort_values("중요도", ascending=False) # 중요도 높은 순으로 정렬
importance
# 결정나무에서는 성별이 0.624로 압도적이었다
# 그런데 숲에서는 요금, 성별, 나이가 고르게 나눠 가진다
# => 나무 한 그루는 매번 가장 강한 특성(성별)부터 물어봐서 성별에 몰린다
# => 반면 숲은 나무마다 특성을 랜덤으로 가져가기 때문에, 
#    성별을 못 보는 나무는 요금, 나이로 갈래를 나눈다
# * 그 결과 "사실 요금과 나이도 생존 여부를 알 수 있는 중요한 정보"라는 것이
#   드러나게 된다
# * 즉, 랜덤 포레스트의 특성 중요도는 한 특성에 쏠리지 않고 더 공정하게
#   각 특성의 기여를 보여준다

### 최적의 나무 수
- 하이퍼파라미터를 고르는 가장 기본적인 방법은 값을 바꿔가며 평가용 성능을 비교

In [ ]:
df = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'sibsp',
             'parch', 'embarked']]

# 전처리
df['sex'] = df['sex'].map({"male":0, "female":1})
df['age'] = df['age'].fillna(df['age'].median())
# 승선도시의 결측값은 최빈값으로 대체, 문자열을 숫자로 변환
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0]).map({"S":0, "C":1, "Q":2})

# 특성(독립변수)/타깃(종속변수) 분할
X = df.drop(columns='survived') # survived를 제외한 7개 특성
y = df['survived']

# 훈련/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

df.head()

In [ ]:
# 랜덤 포레스트에 설정할 나무 개수
tree_counts = [1, 5, 10, 20, 50, 100, 200, 300]
# 나무 수별 정확도를 저장할 리스트
train_acc = [] # 훈련 데이터 정확도
test_acc = [] # 테스트 데이터 정확도

for n in tree_counts :
    # 나무 수만 바꾸고 나머지는 그대로 모델 생성
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))
    print(f"나무 {n}그루 | 학습용={train_acc[-1] : .3f} | 평가용={test_acc[-1] : .3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
# 시각화로 확인

plt.figure(figsize=(7, 4))
plt.plot(tree_counts, train_acc, 'o-', label="학습용", color='#B0C4DE', markersize=5)
plt.plot(tree_counts, test_acc, 's-', label="평가용", color='#3A6B9F', markersize=5)

plt.xscale('log') # 1~300을 고르게 보기위해 x축을 로그 눈금으로
plt.xticks(tree_counts, tree_counts) # 눈금에 실제 나무수를 표시
plt.title("나무 수에 따른 정확도")
plt.xlabel("나무 수")
plt.ylabel("정확도")
plt.legend() # 범례표시
plt.grid(alpha=0.3) # 격자 표시
plt.show()

# 1그루일때 평가용 0.749로 가장 낮다 => 나무가 하나뿐이면 사실상 결정나무
# 나무가 늘 수록 평가용 정확도가 오른다(나무마다 실수가 상쇄)
# 그러다 50그루부터 그래프가 평평해진다(이미 충분히 안정된 것이다)
# ** 나무를 아무리 늘려도 평가용 정확도가 떨어지지 않는다
#    (많이 심는다고 과적합되지 않는다 => 계산 시간만 늘어난다)

## 특성 랜덤(max_features) 
> 각 갈림길마다 특성 후보를 일부만 무작위로 보여줘야 다양한 나무가 된다

In [ ]:
# 각 나무가 맨 처음 던진 질문(뿌리 노드의 질문)이 무엇이였는지
# 세보면, 나무들이 서로 다른지 한눈에 알 수 있다
from collections import Counter # 개수를 세주는 도구

for mf in ['sqrt', None] :
    # max_feature='sqrt'(기본값, 7개 중 2~3개만 봄),
    #            = None(7개를 전부 봄 = 특성 랜덤 끄기)
    forest = RandomForestClassifier(
        n_estimators=100, max_features=mf, random_state=42
    )
    forest.fit(X_train, y_train)
    # 모델들의 첫질문을 저장할 리스트
    first_question = []
    for tree in forest.estimators_ :
        # forest.estimator_ : 숲을 이루는 나무 100그루가 들어있는 리스
        feature_number = tree.tree_.feature[0] # 뿌리에서 사용한 특성번호(독립변수)
        # 번호를 이름으로 바꿔 저장
        first_question.append(X.columns[feature_number])
    print(f"{mf} -> 100그루의 첫 질문 분포")
    print(f"{dict(Counter(first_question).most_common())}")\

# None(특성 랜덤 끄기)일때는 100그루가 전부 성별로 시작한다
# => 나무 100그루가 하나같이 똑같아진 것이다
# 기본값(sqrt)에서는 첫 질문이 성별, 객실등급, 요금으로 골고루 흩어진다
# ** 똑같은 나무 100그루는 똑같은 실수를 100번 반복할 뿐이라 투표의 이점이
#    사라진다
# ** 특별한 이유가 없다면 기본값을 그대로 쓰는 것이 좋다

## GridSearchCV
- 하이퍼파라미터가 여러개면 조합을 하나씩 따져야 한다
- 나무수 3가지 x 특성 랜덤 2가지 = 6가지 조합을 봐야한다
- 하이퍼파라미터 값 후보만 작성하면 모든 조합을 대신 시도해
- 가장 좋은 것을 찾아주는 도구

In [ ]:
from sklearn.model_selection import GridSearchCV

# 하이퍼파라미터 값 후보들을 딕셔너리로 작성
param_grid = {
    "n_estimators" : [100, 200, 300], # 나무 수 3가지
    "max_depth" : [3, 5, 7, None], # 나무 깊이 4가지
    "max_features" : ['sqrt', None] # 특성 랜덤 2가지
}
# cv=5 : 학습 데이터를 5조각으로 나눠 돌아가며 점검(교차 검증)
# n_jobs=-1 : CPU를 모두 써서 빠르게
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                   param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train) # 24가지 조합을 전부 학습 및 테스트한다
# => 시간이 좀 걸린다
print(f"최적 조합 : {grid.best_params_}")
print(f"최적 정확도 : {grid.best_score_ : .3f}")

# 24가지 조합을 전부 시도하고
# 깊이 : 5, 기본 특성 랜덤, 나무 100그루가 가장 좋다고 알려준다.

# 랜덤 포레스트 (Random Forest) 과제

랜덤 포레스트는 조금씩 다르게 학습한 **결정나무 여러 그루의 다수결**로 예측하는 앙상블 모델이다.
이번 과제는 ① 나무 한 그루보다 **숲이 왜 과적합에 강한지**(문제 1), ② **나무 수(`n_estimators`)** 를 늘리면
성능이 어떻게 변하는지(문제 2)를 직접 확인한다.

- 공통 데이터: **`day06_고객이탈.csv`**(통신사 고객 800명, 과제 노트북과 같은 폴더) — 자세한 설명은 코드 주석 참고
- 랜덤 포레스트는 결정나무(day04)를 여러 그루 심은 것이라, **사용법이 결정나무와 거의 똑같다.**
- 각 문제는 `데이터 준비 → 8:2 분할 → 학습 → 평가 → 결과 해석 → 시각화` 흐름을 따른다.

## 문제 1) 나무 한 그루 vs 숲 — 과적합에 강한 건 누구?

고객이 통신사를 **해지(이탈)할지** 맞히는 문제다. 어느 한 가지 이유로 해지하는 게 아니라 요금·약정·문의 횟수가
조금씩 겹쳐 작용하고, 사람 마음은 데이터로 다 설명되지도 않는다. 이런 데이터에서 **나무 한 그루와 숲** 중
누가 더 잘 버티는지 확인하자.

1. `day07_고객이탈.csv`를 불러오고(코드의 데이터 설명 주석 참고), 타깃 `이탈`의 **분포**(`value_counts`)를 확인한다.
2. `X` = `이탈`을 뺀 5개 특성, `y` = `이탈` 로 두고 **8:2 분할**(`random_state=42`)한다.
3. **둘 다 깊이 제한 없이** `DecisionTreeClassifier` 와 `RandomForestClassifier(n_estimators=100)` 을 학습하고,
   각각의 **학습용·평가용 정확도를 모두** 출력한다.
   - 힌트: 학습용 정확도는 `accuracy_score(y_train, 모델.predict(X_train))`
4. 랜덤 포레스트의 **`classification_report`** 를 출력한다.
5. **나이 35 · 월요금 90000 · 통화시간 150 · 문의횟수 4 · 약정개월 3** 인 고객의 이탈 여부를 예측한다.

6. 결과를 **해석**한다. (학습용 정확도는 왜 둘 다 높은가 / 평가용에서 왜 갈리는가 / 숲이 이기는 이유는 무엇인가)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('./day06_고객이탈.csv')
# df.head()
print(df['이탈'].value_counts())

X = df.drop(columns=['이탈'])
y = df['이탈']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

one_tree = DecisionTreeClassifier(random_state=42)
one_tree.fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
print("결정나무 1그루")
print(f"학습용 정확도 : {accuracy_score(y_train, one_tree.predict(X_train)) : .3f}")
print(f"평가용 정확도 : {accuracy_score(y_test, one_tree.predict(X_test)) : .3f}")
print()
print("랜덤 포레스트")
print(f"학습용 정확도 : {accuracy_score(y_train, forest.predict(X_train)) : .3f}")
print(f"평가용 정확도 : {accuracy_score(y_test, forest.predict(X_test)) : .3f}")

print(classification_report(y_test, forest.predict(X_test)))

cu = pd.DataFrame([{
    '나이': 35,
    '월요금': 90000,
    '통화시간': 150,
    '문의횟수': 4,
    '약정개월': 3
}])
pred = forest.predict(cu)
pred_proba = forest.predict_proba(cu)

print("예측 결과 (0: 유지, 1: 이탈):", pred[0])
print(f"이탈 확률: {pred_proba[0][1] * 100:.1f}%")

# 깊이 제한을 두지 않았기 때문에 정확도가 둘다 높다
# 단일 결정나무는 과적합 문제 발생, 랜덤 포레스트 경우 상쇄되면서 결과 반영
# 숲이 이기는 이유 100그루 나무가 투표를 진행, 일반화된 판단으로 보완해줘서

## 문제 2) 나무를 몇 그루 심을까? — 나무 수와 특성 중요도

숲이 나무 한 그루보다 낫다는 것은 확인했다. 그렇다면 **몇 그루**를 심어야 할까? 많이 심을수록 계속 좋아질까?
나무 수를 바꿔가며 직접 확인하고, 숲이 알려주는 **특성 중요도**도 읽어보자.

1. 같은 `day06_고객이탈.csv`를 불러와 **8:2 분할**(`random_state=42`)한다.
2. **`n_estimators`를 1 · 3 · 5 · 10 · 20 · 50 · 100 · 200** 으로 바꿔가며 학습하고,
   각 경우의 **학습용·평가용 정확도**를 구한다. (반복문)
3. 나무 수별 **학습용·평가용 정확도를 하나의 꺾은선 그래프**로 그린다.
4. **평가용 정확도가 가장 높은 나무 수**를 찾아 출력한다.
5. 나무 100그루 숲의 **특성 중요도**(`feature_importances_`)를 표로 출력
6. (도전) `max_features`를 **1 / `"sqrt"`(기본) / `None`(전부 사용)** 으로 바꿔 평가용 정확도를 비교한다.
   - 힌트: `None`은 갈림길마다 5개 특성을 **전부** 본다는 뜻 = 수업 2번의 **"특성 랜덤"을 꺼버리는 것**
7. 결과를 **해석**한다. (나무가 1그루면 왜 불안정한가 / 어디서부터 평평해지나 / 많이 심으면 과적합되나 /
   어떤 특성이 이탈을 가르나 / `max_features=None`이 왜 더 나쁜가)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

plt.rcParams['font.family'] = 'Malgun Gothic'

df = pd.read_csv('./day06_고객이탈.csv')
# df.head()

X = df.drop(columns=['이탈'])
y = df['이탈']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
n_list = [1, 3, 5, 10, 20, 50, 100, 200]
train_acc = []
test_acc = []

for n in n_list :
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)

    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score = accuracy_score(y_test, model.predict(X_test))

    train_acc.append(train_score)
    test_acc.append(test_score)
    print(f"나무 : {n:3d} | 학습용 : {train_acc[-1] : .3f} | 평가용 : {test_acc[-1] : .3f}")

plt.figure(figsize=(10, 5))
plt.plot(n_list, train_acc, marker='o', label='학습용 정확도')
plt.plot(n_list, test_acc, marker='o', label='평가용 정확도')
plt.title('나무수에 따른 정확도 변화')
plt.xlabel("나무수")
plt.ylabel("정확도")
plt.xticks(n_list)
plt.legend()
plt.grid()
plt.show()

best_score = max(test_acc)
best_n = n_list[test_acc.index(best_score)]

print(f"가장 높은 나무 수 : {best_n} | 가장 높은 정확도 : {best_score}")

forest_100 = RandomForestClassifier(n_estimators=100, random_state=42)
forest_100.fit(X_train, y_train)
forest_df = pd.DataFrame({
    "특성" : X.columns,
    "중요도" : forest_100.feature_importances_
}).sort_values("중요도", ascending=False)

print(forest_df)

mf_list = [1, 'sqrt', None]

for mf in mf_list:
    model = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)
    model.fit(X_train, y_train)
    print(f"max_features={str(mf):.5s} | 평가용 정확도={accuracy_score(y_test, model.predict(X_test))}")

# 나무가 1그루면 과적합, 20그루 이후 부터 평평해진다. 
# 많이 심는다고 과적합 되지 않는다 시간만 늘어날 뿐!
# 월요금, 약정개월이, 고객센터 문의 빈도로 이탈이 갈린다
# None으로 주면 무작위성이 사라져서 성능이 줄어든다